# Step 4: Statistical tests. Does late delivery hurt reviews and repeat purchases?

**Business question:** Why are some orders delivered late, and how does lateness affect review scores and repeat purchases?

**Hypotheses**
1. **H1:** Late orders get lower review scores than on-time orders.
2. **H2:** Review scores fall as the delay grows (dose-response).
3. **H3:** Customers whose first order was late are less likely to place a second order.

**Method notes**
- Review score is ordinal (1 to 5) and skewed, so I use rank-based tests (Mann-Whitney U, Spearman, Kruskal-Wallis) instead of t-tests.
- Repeat purchase is binary, so I use a chi-square test on a 2x2 table.
- With about 90K rows, p-values will be tiny almost regardless of effect size. So every test also reports an **effect size** and a **confidence interval**.

**Input:** `order_analysis` exported from PostgreSQL as CSV (one row per delivered order).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.3f}".format)
RNG = np.random.default_rng(42)

candidates = [
    Path("../dashboard/data/order_analysis.csv"),
    Path("../data/order_analysis.csv"),
    Path("order_analysis.csv"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
assert DATA_PATH is not None, (
    "Export the order_analysis table from PostgreSQL to CSV and save it as "
    "dashboard/data/order_analysis.csv"
)

date_cols = ["order_purchase_timestamp", "order_delivered_customer_date",
             "order_estimated_delivery_date", "purchase_month"]
df = pd.read_csv(DATA_PATH, parse_dates=date_cols)
print("Loaded:", DATA_PATH, df.shape)
print(df.head(3).T)

In [ ]:
# Sanity checks before any test
print(df.dtypes)
print()
print("Columns with missing values:")
print(df.isna().sum()[lambda s: s > 0])
print()
print("Duplicate order_id:", df["order_id"].duplicated().sum())
print(f"Overall late rate: {df['is_late'].mean():.2%}")

rev = df.dropna(subset=["review_score"]).copy()
rev["review_score"] = rev["review_score"].astype(int)
rev["status"] = np.where(rev["is_late"] == 1, "Late", "On time")
print(f"{len(df) - len(rev):,} orders without a review excluded from review analysis")

## 1. Review score distribution: late vs on time
Always plot before testing. The shape of the data decides which test is valid.

In [ ]:
dist = (rev.groupby("status")["review_score"]
           .value_counts(normalize=True).mul(100)
           .unstack(0).round(1))
print("% of orders giving each score")
print(dist)

ax = dist.plot(kind="bar", figsize=(8, 4.5), width=0.8)
ax.set_xlabel("Review score")
ax.set_ylabel("% of orders")
ax.set_title("Review score distribution: late vs on-time deliveries")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. H1: Do late orders get lower reviews?
**Test:** Mann-Whitney U (two independent groups, ordinal outcome).
**Effect size:** the common-language effect size (CLES), the probability that a random on-time order outscores a random late one (0.5 means no difference), plus a bootstrap confidence interval for the difference in mean score.

In [ ]:
late = rev.loc[rev["is_late"] == 1, "review_score"]
ontime = rev.loc[rev["is_late"] == 0, "review_score"]

summary = pd.DataFrame({
    "n": [len(ontime), len(late)],
    "mean": [ontime.mean(), late.mean()],
    "median": [ontime.median(), late.median()],
    "pct_1_star": [(ontime == 1).mean() * 100, (late == 1).mean() * 100],
    "pct_4_5_star": [(ontime >= 4).mean() * 100, (late >= 4).mean() * 100],
}, index=["On time", "Late"])
print(summary)

u_stat, p_mw = stats.mannwhitneyu(ontime, late, alternative="two-sided")
cles = u_stat / (len(ontime) * len(late))
rank_biserial = 2 * cles - 1
print(f"\nMann-Whitney U = {u_stat:,.0f}, p = {p_mw:.3e}")
print(f"CLES = {cles:.3f}  (0.5 = no difference)")
print(f"Rank-biserial correlation = {rank_biserial:.3f}")

# Bootstrap CI for the difference in means, resampling the 1-5 score counts (fast)
scores = np.arange(1, 6)
def boot_mean(series, n_boot=5000):
    counts = series.value_counts().reindex(scores, fill_value=0).to_numpy()
    probs = counts / counts.sum()
    draws = RNG.multinomial(counts.sum(), probs, size=n_boot)
    return (draws * scores).sum(axis=1) / counts.sum()

diff_samples = boot_mean(ontime) - boot_mean(late)
ci_low, ci_high = np.percentile(diff_samples, [2.5, 97.5])
mean_gap = ontime.mean() - late.mean()
print(f"Mean score gap (on time - late) = {mean_gap:.2f}  95% CI [{ci_low:.2f}, {ci_high:.2f}]")

## 3. H2: Does the review score fall as the delay grows?
**Test:** Spearman rank correlation between `delay_days` and `review_score` (works for ordinal data and monotonic relationships).
Then a dose-response view: average score by exact days of delay.

In [ ]:
rho_all, p_all = stats.spearmanr(rev["delay_days"], rev["review_score"])
late_only = rev[rev["is_late"] == 1]
rho_late, p_late = stats.spearmanr(late_only["delay_days"], late_only["review_score"])
print(f"All delivered orders: Spearman rho = {rho_all:.3f}, p = {p_all:.2e}")
print(f"Late orders only    : Spearman rho = {rho_late:.3f}, p = {p_late:.2e}")

# Dose-response: mean score by days relative to the estimated date
tmp = rev.assign(delay_clip=rev["delay_days"].clip(-20, 30))
binned = tmp.groupby("delay_clip")["review_score"].agg(["mean", "count"])
binned = binned[binned["count"] >= 30]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(binned.index, binned["mean"], marker="o", linewidth=1.5)
ax.axvline(0, color="red", linestyle="--", label="Estimated delivery date")
ax.set_xlabel("Days vs estimated delivery date (negative = early, clipped at -20 / +30)")
ax.set_ylabel("Average review score")
ax.set_title("Review score by delivery delay")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Same idea with the delay buckets from SQL, plus an overall test across buckets
groups = [g["review_score"].to_numpy() for _, g in rev.groupby("delay_bucket")]
h_stat, p_kw = stats.kruskal(*groups)
print(f"Kruskal-Wallis H = {h_stat:,.1f}, p = {p_kw:.2e}")

bucket = rev.groupby("delay_bucket")["review_score"].agg(n="count", mean="mean", sem="sem")
bucket["ci95"] = 1.96 * bucket["sem"]
print(bucket)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(range(len(bucket)), bucket["mean"], yerr=bucket["ci95"], capsize=4)
ax.set_xticks(range(len(bucket)))
ax.set_xticklabels(bucket.index, rotation=15)
ax.set_ylabel("Average review score (95% CI)")
ax.set_ylim(1, 5)
ax.set_title("Review score by delay bucket")
plt.tight_layout()
plt.show()

## 4. H3: Do customers with a late first order reorder less?
**Test:** chi-square test of independence on a 2x2 table (first order late or not vs reordered or not).
Only first orders placed before 1 Feb 2018 are used, so every customer had at least about 6 months to come back.
**Effect sizes:** difference in repeat rate (percentage points) with a 95% CI, relative risk, and phi.
Note: repeat purchase is rare in this dataset, so the effect may be small.

In [ ]:
ordered = df.sort_values("order_purchase_timestamp")
totals = ordered.groupby("customer_unique_id").size().rename("total_orders")
first = (ordered.drop_duplicates("customer_unique_id", keep="first")
                .merge(totals, on="customer_unique_id"))
first = first[first["order_purchase_timestamp"] < "2018-02-01"].copy()
first["repeat"] = (first["total_orders"] > 1).astype(int)

ct = pd.crosstab(first["is_late"], first["repeat"])
ct.index = ["First order on time", "First order late"]
ct.columns = ["No reorder", "Reordered"]
print(ct)

chi2, p_chi, dof, expected = stats.chi2_contingency(ct, correction=False)
print(f"\nMinimum expected cell count: {expected.min():.1f}  (should be > 5)")

n_on, n_late = ct.iloc[0].sum(), ct.iloc[1].sum()
r_on = ct.iloc[0, 1] / n_on
r_late = ct.iloc[1, 1] / n_late
diff = r_late - r_on
se = np.sqrt(r_on * (1 - r_on) / n_on + r_late * (1 - r_late) / n_late)
rel_risk = r_late / r_on
phi = np.sqrt(chi2 / ct.values.sum())

print(f"Repeat rate, first order on time: {r_on:.2%}")
print(f"Repeat rate, first order late   : {r_late:.2%}")
print(f"Difference (late - on time)     : {diff * 100:.2f} pp  "
      f"95% CI [{(diff - 1.96 * se) * 100:.2f}, {(diff + 1.96 * se) * 100:.2f}]")
print(f"Relative risk                   : {rel_risk:.2f}")
print(f"Chi-square = {chi2:.2f}, dof = {dof}, p = {p_chi:.3g}, phi = {phi:.3f}")

## 5. Robustness: is it just a geography effect?
Late deliveries cluster in certain states. If the review gap only appeared in one region, the effect could be about that region and not about lateness. Check the gap inside each of the biggest states.

In [ ]:
top_states = rev["customer_state"].value_counts().head(6).index
rows = []
for s in top_states:
    sub = rev[rev["customer_state"] == s]
    a = sub.loc[sub["is_late"] == 0, "review_score"]
    b = sub.loc[sub["is_late"] == 1, "review_score"]
    if len(b) < 50:
        continue
    _, p = stats.mannwhitneyu(a, b)
    rows.append({"state": s, "n_late": len(b), "mean_on_time": a.mean(),
                 "mean_late": b.mean(), "gap": a.mean() - b.mean(), "p_value": f"{p:.2e}"})
print(pd.DataFrame(rows).round(3).to_string(index=False))

## 6. Summary for the README
The cell below prints the headline numbers. Copy them into the README findings and the resume bullets.

In [ ]:
print("KEY FINDINGS")
print("-" * 60)
print(f"H1  Mean review: on time {ontime.mean():.2f} vs late {late.mean():.2f} "
      f"(gap {mean_gap:.2f}, 95% CI [{ci_low:.2f}, {ci_high:.2f}]); "
      f"Mann-Whitney p = {p_mw:.2e}; CLES = {cles:.2f}")
print(f"H1  1-star share: on time {(ontime == 1).mean():.1%} vs late {(late == 1).mean():.1%}")
print(f"H2  Spearman rho (delay vs review) = {rho_all:.2f}; Kruskal-Wallis p = {p_kw:.2e}")
print(f"H3  Repeat rate after on-time first order {r_on:.2%} vs late {r_late:.2%} "
      f"(diff {diff * 100:.2f} pp, chi-square p = {p_chi:.3g})")

### How to read the results (replace with your own words)
- **H1:** State the gap in plain English, for example "late orders average X vs Y stars", and say whether it is practically large (look at CLES and the 1-star share), not only significant.
- **H2:** Say where the drop is steepest. That point is your threshold for proactive customer messaging.
- **H3:** If the repeat-rate difference is small or the CI includes 0, say so. An honest "late delivery hurts satisfaction but I could not show a clear effect on repeat purchase in this data" is a stronger finding than an exaggerated claim.

### Limitations
- **Association, not causation.** Late orders also differ in region, category and seller, so the gap may include those effects (see section 5).
- **Review bias.** Only customers who chose to leave a review are included, and unhappy customers review more often.
- **Repeat purchase is rare (about 3%)**, so the test for H3 has low power.
- **Data covers 2016 to 2018** from one marketplace in Brazil, so results may not generalize.